# Assignment 4 - Multimodal Interactions
**CS 5970/6970 | Spring 2026**

This notebook builds an agentic system that uses CLIP as a retrieval (RAG) system over images and BLIP for visual grounding via captioning.

## Stage 0: Environment setup

In [ ]:
import os

REPO_URL = "https://github.com/dutch-casa/gen-ai-project-four.git"
REPO_DIR = "gen-ai-project-four"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print(f"Cloned {REPO_URL}")
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q torch torchvision transformers ftfy regex Pillow

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import os
import json
import random
import numpy as np
import torch
import clip
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 3232004
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Seed: {SEED}")

## Stage 1: Dataset setup

In [ ]:
IMAGE_DIR = "assignment4_images"

image_paths = sorted([
    os.path.join(IMAGE_DIR, f"image_{i:03d}.jpg")
    for i in range(100)
])

image_ids = [f"image_{i:03d}.jpg" for i in range(100)]

print(f"Loaded {len(image_paths)} image paths")
print(f"First: {image_paths[0]}")
print(f"Last:  {image_paths[-1]}")

missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    print(f"WARNING: {len(missing)} images missing!")
else:
    print("All 100 images found.")

## Stage 2: CLIP as a RAG system

In [ ]:
print("Loading CLIP model:")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print("CLIP ViT-B/32 loaded.")

In [ ]:
print("Computing image embeddings:")

image_embeddings = []

with torch.no_grad():
    for i, path in enumerate(image_paths):
        img = clip_preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        emb = clip_model.encode_image(img)
        image_embeddings.append(emb)
        if (i + 1) % 25 == 0:
            print(f"  {i + 1}/100 images embedded")

image_embeddings = torch.cat(image_embeddings, dim=0)
image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)

print(f"Embedding matrix shape: {image_embeddings.shape}")

In [ ]:
def clip_retrieve(query: str, top_k: int = 5) -> list:
    """
    Retrieve the top-k most relevant images for a text query.
    Returns list of (image_id, score) tuples.
    """
    text_tokens = clip.tokenize([query]).to(device)

    with torch.no_grad():
        text_emb = clip_model.encode_text(text_tokens)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    similarities = (image_embeddings @ text_emb.T).squeeze(1)
    top_indices = similarities.argsort(descending=True)[:top_k]

    results = []
    for idx in top_indices:
        results.append((image_ids[idx.item()], similarities[idx].item()))

    return results

In [ ]:
print("Stage 2 retrieval demo (5 queries, top-5 each):")

demo_queries = [
    "a photo of a dog",
    "a vehicle on a road",
    "people playing sports",
    "cooking food in a kitchen",
    "outdoor scene with people"
]

for query in demo_queries:
    results = clip_retrieve(query, top_k=5)
    print(f'\nQuery: "{query}"')
    for img_id, score in results:
        print(f"  {img_id}  score={score:.4f}")

## Stage 3: Image captioning (BLIP)

In [ ]:
print("Loading BLIP model (half-precision):")

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16
).to(device)

print("BLIP loaded in float16.")

In [ ]:
def caption_image(image_path: str) -> str:
    """
    Generate a one-sentence caption for a single image.
    """
    img = Image.open(image_path).convert("RGB")
    inputs = blip_processor(images=img, return_tensors="pt").to(device, torch.float16)

    with torch.no_grad():
        output_ids = blip_model.generate(**inputs, max_new_tokens=50)

    caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)
    return caption

In [ ]:
print("Stage 3 captioning demo (5 images):")

demo_image_indices = [0, 20, 40, 60, 80]
for idx in demo_image_indices:
    path = image_paths[idx]
    cap = caption_image(path)
    print(f"  {image_ids[idx]}: {cap}")

## Stage 4: Agentic RAG pipeline

In [ ]:
def agentic_query(query: str, top_k: int = 5, caption_top: int = 3) -> dict:
    """
    Single-turn agentic pipeline:
      1. Decide if this is a direct image inspection or a retrieval query.
      2. Retrieve with CLIP if needed.
      3. Caption selected images with BLIP.
      4. Produce a final answer grounded in the captions.
    """
    tool_trace = []

    # Check if the query references a specific image by filename
    direct_image = None
    for img_id in image_ids:
        if img_id in query:
            direct_image = img_id
            break

    if direct_image:
        # Direct inspection: skip retrieval, just caption the named image
        path = os.path.join(IMAGE_DIR, direct_image)
        cap = caption_image(path)
        tool_trace.append(f"caption_image({direct_image})")

        return {
            "query": query,
            "retrieved_images": [direct_image],
            "selected_image": direct_image,
            "caption": cap,
            "tool_trace": tool_trace,
            "final_answer": f"{direct_image}: {cap}"
        }

    # Retrieval path
    retrieved = clip_retrieve(query, top_k=top_k)
    tool_trace.append(f"clip_retrieve('{query}', top_k={top_k})")

    retrieved_ids = [img_id for img_id, _ in retrieved]

    # Caption the top candidates (up to caption_top, max 5)
    captions = {}
    for img_id, score in retrieved[:caption_top]:
        path = os.path.join(IMAGE_DIR, img_id)
        cap = caption_image(path)
        captions[img_id] = cap
        tool_trace.append(f"caption_image({img_id})")

    # Pick the best image: re-rank by checking if the caption
    # relates to the query via CLIP text-text similarity
    best_image = retrieved_ids[0]
    best_score = -1.0

    with torch.no_grad():
        query_tokens = clip.tokenize([query]).to(device)
        query_emb = clip_model.encode_text(query_tokens)
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)

        for img_id, cap in captions.items():
            cap_tokens = clip.tokenize([cap]).to(device)
            cap_emb = clip_model.encode_text(cap_tokens)
            cap_emb = cap_emb / cap_emb.norm(dim=-1, keepdim=True)
            sim = (query_emb @ cap_emb.T).item()
            if sim > best_score:
                best_score = sim
                best_image = img_id

    best_caption = captions.get(best_image, "")

    # Build the final answer from grounded evidence
    final_answer = (
        f"Retrieved {len(retrieved_ids)} candidates. "
        f"Best match: {best_image}. "
        f"Caption: {best_caption}"
    )

    return {
        "query": query,
        "retrieved_images": retrieved_ids,
        "selected_image": best_image,
        "caption": best_caption,
        "tool_trace": tool_trace,
        "final_answer": final_answer
    }

## Stage 5: Query evaluation (15 queries)

In [ ]:
queries = [
    # A. Retrieval queries
    "Find me a picture of a dog.",
    "Find me a picture of a vehicle.",
    "Find me an image showing sports.",
    "Find me an image that likely involves cooking.",
    "Find me an outdoor scene with people.",
    # B. Direct image inspection
    "Here is the path to image_005.jpg. Describe it.",
    "Here is the path to image_021.jpg. What is happening?",
    "Describe image_044.jpg in one sentence.",
    "What objects are visible in image_073.jpg?",
    "Describe image_090.jpg briefly.",
    # C. Retrieval + grounding
    "Find me a picture of people eating and describe it.",
    "Find me an image of a person riding something and summarize it.",
    "Find an image that appears to show a kitchen scene and explain why.",
    "Find an image with animals outdoors and describe the scene.",
    'Find the best image for "family meal" and describe it.'
]

print(f"Running {len(queries)} queries:")

In [ ]:
all_results = []

for i, q in enumerate(queries, 1):
    print(f"\nQuery {i}: {q}")
    result = agentic_query(q)
    all_results.append(result)
    print(f"  Retrieved: {result['retrieved_images']}")
    print(f"  Selected:  {result['selected_image']}")
    print(f"  Caption:   {result['caption']}")
    print(f"  Tools:     {result['tool_trace']}")
    print(f"  Answer:    {result['final_answer']}")

In [ ]:
print("Full JSON output:")
print(json.dumps(all_results, indent=2))

## Stage 6: Insight blocks

### Block 1: CLIP as RAG

CLIP worked well for concrete, single-object queries. "A photo of a dog" returned image_001, image_007, and image_000 as top hits, and BLIP confirmed these all contained dogs. "A vehicle on a road" pulled image_013 and image_015, with BLIP confirming "a car parked in front of a white bus" for the selected result. The similarity scores were spread out enough that ranking felt meaningful.

It got shakier with situational queries. "Cooking food in a kitchen" ranked image_035 highest by CLIP score, but the re-ranking step correctly promoted image_078 after BLIP captioned it as "a man in a kitchen preparing food." CLIP retrieved the right neighborhood but not the right top-1. For "outdoor scene with people," it picked image_004 as the CLIP top result, but the re-ranker selected image_054 instead. BLIP then captioned that as "a man is throwing a frurd frck," which is gibberish. CLIP got the gist right, but the pipeline downstream failed to catch the bad caption.

### Block 2: Retrieval vs captioning

Retrieval alone was sufficient for queries 1-3 (dog, vehicle, sports). CLIP's top results already matched the intent, and captioning just confirmed what the scores already indicated. For query 3, CLIP ranked image_026 first, but BLIP's caption for image_028 ("a baseball player is getting ready to hit the ball") was a stronger semantic match to "sports," so re-ranking helped. That said, both were sports images. Captioning didn't change the category, just picked a better representative.

Captioning was necessary for queries 4 and 13 (cooking, kitchen scene). Both times, CLIP's top-1 was not the best answer. The re-ranking step, which compares BLIP's caption text against the query via CLIP embeddings, promoted image_078 ("a man in a kitchen preparing food") over alternatives that CLIP scored higher visually. Without captioning, the system would have returned a less relevant image. Captioning also mattered for query 12 (person riding something), where it correctly promoted image_087 ("a person riding a horse across a field") over image_022, which CLIP had ranked first.

### Block 3: Agentic behavior

The system routed queries correctly. For queries 6-10 (direct inspection), it detected the filename in the query string, skipped CLIP retrieval, and called only `caption_image`. The tool traces for those queries show a single tool call each. For retrieval queries, it called `clip_retrieve` first, then `caption_image` on the top 3 candidates, producing 4 tool calls per query.

Effective tool use: Query 13 ("kitchen scene"). CLIP's top-1 was image_039, but after captioning three candidates, the re-ranker selected image_078 ("a man in a kitchen preparing food"). The caption directly addressed the query. Retrieval found the right neighborhood, captioning picked the right image.

Unnecessary tool use: Query 1 ("picture of a dog"). CLIP returned five images, all plausibly dogs. The system captioned three of them. It ended up selecting image_007 over image_001 based on caption re-ranking, but both were dogs. The captioning step burned three BLIP inference calls to make a lateral move. For simple, concrete queries, the CLIP top-1 was already good enough.

### Block 4: Failure case

Query 14 ("Find an image with animals outdoors and describe the scene") is the clearest failure. CLIP retrieved image_051, image_060, and image_082 as top candidates. BLIP then captioned image_082 as "a gi gi gi gi gi gi gi gi gi..." repeating indefinitely until the token limit. This is a known degenerate behavior in BLIP under half-precision: certain images trigger repetition loops in the decoder.

The failure is on the captioning side, not retrieval. CLIP found relevant animal images. But the re-ranking step compared that gibberish caption against the query embedding and, bizarrely, scored it highest. The system has no check for degenerate outputs. A simple heuristic like detecting repeated tokens would have caught this and fallen back to the next candidate. Query 5 had the same issue: image_054 was captioned as "a man is throwing a frurd frck," which is nonsense but still won the re-ranking.

### Block 5: Grounding vs hallucination

Every final answer is grounded in either CLIP retrieval scores or BLIP captions. The system never invents information beyond what those two models produce. For the direct inspection queries (6-10), the answer is literally the BLIP caption. For retrieval queries, the answer cites the selected image and its caption.

That said, the system inherits BLIP's hallucinations. Query 14's "a gi gi gi..." and query 5's "a frurd frck" are both model-generated nonsense that the pipeline passed through without question. These are not hallucinations in the sense of fabricated facts. They are degenerate decoder outputs. But the effect is the same: the final answer contains text that has no relationship to the image content.

BLIP also produced vague-but-true captions in some cases. Query 11 selected image_031 with the caption "a group of people sitting around a table," which matched the query about people eating. The caption doesn't actually confirm anyone is eating. That's a softer grounding failure: the system treated proximity (people + table) as confirmation of the action (eating). The visual evidence supports the answer loosely but not precisely.